# 42 — Phái sinh và chênh lệch với chỉ số cơ sở

`VN30F1M` là hợp đồng tương lai chỉ số VN30 tháng gần nhất. Ba điều làm nó khác
mọi thứ bạn đã gặp trong 41 notebook trước:

1. **Đơn vị là điểm chỉ số và hợp đồng**, không phải nghìn VND và cổ phiếu
2. **`basis` không có `interval`** — và đó là một quyết định thiết kế có lý do
3. **Chỉ hai nhóm nhà đầu tư** tồn tại, và kiểu của tham số chặn từ lúc gõ code

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import finlens
from finlens_examples import ap_dung_theme, duong, hom_nay, lui_ngay, nen, thanh_doi_mau, ty_dong
from finlens_examples.charts import CHUOI, GIAM, TANG

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

HD = "VN30F1M"

## 1 · ⚠️ Đơn vị: điểm chỉ số và hợp đồng

Đây là namespace thứ ba với đơn vị riêng. Nhân giá phái sinh với 1.000 để "ra
VND" — phản xạ đúng với cổ phiếu — là sai hoàn toàn ở đây.

In [2]:
fut = client.eod.derivative.ohlcv(HD, start=lui_ngay(HOM_NAY, nam=1))

print("Đơn vị của phái sinh:")
for cot, dv in fut.attrs["finlens"]["units"].items():
    if dv:
        print(f"  {cot:<8} {dv}")

co_phieu = client.eod.stock.ohlcv("HPG", start=lui_ngay(HOM_NAY, ngay=10))
print(f"\nĐối chiếu — cổ phiếu: close={co_phieu.attrs['finlens']['units']['close']}, "
      f"volume={co_phieu.attrs['finlens']['units']['volume']}")

Đơn vị của phái sinh:
  open     index_point
  high     index_point
  low      index_point
  close    index_point
  volume   contract



Đối chiếu — cổ phiếu: close=kVND, volume=share


| Loại | Cột giá | Khối lượng |
|---|---|---|
| Cổ phiếu | `kVND` — nghìn VND | `share` — cổ phiếu |
| Chỉ số | `index_point` | `share` |
| **Phái sinh** | **`index_point`** | **`contract`** |
| Chứng quyền | `VND` thô | `warrant` |

Giá `1928.6` của `VN30F1M` là **1.928,6 điểm chỉ số**. Giá trị một hợp đồng là
điểm nhân với hệ số nhân của sản phẩm (100.000 VND/điểm với VN30F1M) — con số
đó **không nằm trong dữ liệu**, nó là đặc tả sản phẩm.

In [3]:
HE_SO_NHAN = 100_000  # VND mỗi điểm chỉ số — đặc tả sản phẩm, không phải dữ liệu

gia_cuoi = fut["close"].iloc[-1]
print(f"Giá đóng cửa gần nhất : {gia_cuoi:,.1f} điểm")
print(f"Giá trị 1 hợp đồng    : {gia_cuoi * HE_SO_NHAN:,.0f} VND")
print(f"Khối lượng phiên      : {fut['volume'].iloc[-1]:,.0f} hợp đồng")
print(f"Giá trị giao dịch     : {ty_dong(fut['volume'].iloc[-1] * gia_cuoi * HE_SO_NHAN)} tỷ đồng")

Giá đóng cửa gần nhất : 1,924.0 điểm
Giá trị 1 hợp đồng    : 192,400,000 VND
Khối lượng phiên      : 230,652 hợp đồng
Giá trị giao dịch     : 44,377.4 tỷ đồng


In [4]:
nen(
    fut.tail(120),
    tieu_de=f"{HD} — 120 phiên gần nhất",
    phu_de="Trục giá là ĐIỂM CHỈ SỐ, trục dưới là HỢP ĐỒNG — không phải nghìn VND và cổ phiếu",
    nhan_gia="điểm chỉ số",
)

## 2 · `basis` — chênh lệch phái sinh với chỉ số cơ sở

```
basis     = future_close − spot_close    # điểm chỉ số
basis_pct = basis / spot_close × 100     # phần trăm, thang 0–100
```

⚠️ **Mẫu số là giá CHỈ SỐ, không phải giá hợp đồng.** Hai mẫu số chỉ lệch nhau
khoảng 0,3% nên chọn nhầm gần như không nhìn ra được — nhưng nó vẫn là một con
số khác.

In [5]:
basis = client.eod.derivative.basis(HD, start=lui_ngay(HOM_NAY, nam=1))

print(f"{len(basis)} phiên · đơn vị: {[f'{k}={v}' for k, v in basis.attrs['finlens']['units'].items() if v]}")
basis.tail(5)

250 phiên · đơn vị: ['future_close=index_point', 'spot_close=index_point', 'basis=index_point', 'basis_pct=pct']


,symbol,index_symbol,date,future_close,spot_close,basis,basis_pct
245,VN30F1M,VN30,2026-08-05,1913.8,1916.88,-3.08,-0.1607
246,VN30F1M,VN30,2026-08-06,1896.0,1902.79,-6.79,-0.3568
247,VN30F1M,VN30,2026-08-07,1890.1,1911.09,-20.99,-1.0983
248,VN30F1M,VN30,2026-08-10,1928.6,1925.18,3.42,0.1776
249,VN30F1M,VN30,2026-08-11,1924.0,1925.82,-1.82,-0.0945


In [6]:
# Kiểm chứng công thức, kể cả mẫu số
tinh_lai = basis["future_close"] - basis["spot_close"]
print(f"basis khớp với (future − spot)? {np.allclose(tinh_lai, basis['basis'], atol=1e-6)}")

theo_spot = basis["basis"] / basis["spot_close"] * 100
theo_fut = basis["basis"] / basis["future_close"] * 100
print(f"basis_pct khớp với mẫu số SPOT?   {np.allclose(theo_spot, basis['basis_pct'], atol=1e-4)}")
print(f"basis_pct khớp với mẫu số FUTURE? {np.allclose(theo_fut, basis['basis_pct'], atol=1e-4)}")
print(f"\nHai mẫu số lệch nhau trung bình {(theo_spot - theo_fut).abs().mean():.4f} điểm phần trăm")
print("→ nhỏ tới mức chọn nhầm không ai phát hiện được. Đó là lý do phải hỏi dữ liệu.")

basis khớp với (future − spot)? True
basis_pct khớp với mẫu số SPOT?   True
basis_pct khớp với mẫu số FUTURE? False

Hai mẫu số lệch nhau trung bình 0.0010 điểm phần trăm
→ nhỏ tới mức chọn nhầm không ai phát hiện được. Đó là lý do phải hỏi dữ liệu.


### Vì sao `basis()` **không** có tham số `interval`

Mọi hàm `ohlcv` đều có `interval`. `basis` thì không, và đây là chủ ý: **gộp
một chênh lệch qua nhiều bước không có nghĩa hiển nhiên nào.**

Basis của một tuần là gì? Trung bình của năm phiên? Giá trị phiên cuối? Chênh
lệch giữa giá cuối tuần của hai chuỗi? Ba câu trả lời khác nhau, không cái nào
hiển nhiên đúng. Thư viện từ chối đoán hộ bạn.

In [7]:
try:
    client.eod.derivative.basis(HD, start=lui_ngay(HOM_NAY, thang=3), interval="1w")
except TypeError as e:
    print(f"TypeError: {e}")

print("\nMuốn basis theo tuần thì bạn tự chọn phép gộp — và nói rõ mình chọn cái nào:")
tuan = (
    basis.assign(tuan=basis["date"].dt.to_period("W"))
    .groupby("tuan", observed=True)
    .agg(basis_trung_binh=("basis", "mean"), basis_cuoi_tuan=("basis", "last"))
    .tail(6)
    .round(2)
)
print(tuan.to_string())

TypeError: EodDerivative.basis() got an unexpected keyword argument 'interval'

Muốn basis theo tuần thì bạn tự chọn phép gộp — và nói rõ mình chọn cái nào:


                       basis_trung_binh  basis_cuoi_tuan
tuan                                                    
2026-07-06/2026-07-12              4.18             5.68
2026-07-13/2026-07-19             -0.97            -1.65
2026-07-20/2026-07-26              4.52             5.64
2026-07-27/2026-08-02              1.52             2.93
2026-08-03/2026-08-09             -7.38           -20.99
2026-08-10/2026-08-16              0.80            -1.82


## 3 · Đọc basis

Basis dương (*contango*) — phái sinh đắt hơn cơ sở, thị trường kỳ vọng tăng
hoặc chi phí nắm giữ dương. Basis âm (*backwardation*) — phái sinh rẻ hơn, kỳ
vọng giảm hoặc có áp lực bán phòng hộ.

In [8]:
gan = basis.tail(60)
print(f"60 phiên gần nhất — basis trung bình {gan['basis'].mean():+.2f} điểm "
      f"({gan['basis_pct'].mean():+.3f}%)")
print(f"Số phiên dương: {(gan['basis'] > 0).sum()}/{len(gan)}")
print(f"Biên độ: {gan['basis'].min():+.1f} → {gan['basis'].max():+.1f} điểm")

thanh_doi_mau(
    gan.assign(nhan=gan["date"].dt.strftime("%d/%m")).tail(30),
    x="nhan",
    y="basis",
    tieu_de=f"{HD} — chênh lệch với VN30, 30 phiên gần nhất",
    phu_de="Dương = phái sinh đắt hơn cơ sở (contango) · âm = rẻ hơn (backwardation)",
    nhan_y="điểm chỉ số",
    dinh_dang_nhan="{:+.1f}",
)

60 phiên gần nhất — basis trung bình +0.99 điểm (+0.050%)
Số phiên dương: 39/60
Biên độ: -21.0 → +11.1 điểm


### Basis và diễn biến chỉ số — hai khung, không phải hai trục y

In [9]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.07, row_heights=[0.6, 0.4])

fig.add_trace(
    go.Scatter(x=basis["date"], y=basis["spot_close"], name="VN30 (cơ sở)",
               line=dict(width=2, color=CHUOI[0])),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=basis["date"], y=basis["future_close"], name=f"{HD} (phái sinh)",
               line=dict(width=2, color=CHUOI[1])),
    row=1, col=1,
)
fig.add_trace(
    go.Bar(x=basis["date"], y=basis["basis"], name="Basis",
           marker=dict(color=[TANG if v >= 0 else GIAM for v in basis["basis"]], line=dict(width=0))),
    row=2, col=1,
)
fig.add_hline(y=0, line_width=1, line_color="#c3c2b7", row=2, col=1)
fig.update_yaxes(title_text="điểm chỉ số", row=1, col=1)
fig.update_yaxes(title_text="basis (điểm)", row=2, col=1)
fig.update_layout(
    title_text=f"{HD} và VN30 — 12 tháng<br>"
    "<sub style='color:#52514e'>Hai đường trên gần như trùng nhau; khung dưới phóng to đúng phần chênh lệch</sub>",
    height=680,
    hovermode="x unified",
)
fig

Hai đường ở khung trên gần như chồng lên nhau — đó chính là lý do khung dưới
tồn tại. Chênh lệch cỡ vài điểm trên nền 1.900 điểm là **0,1%**; không mắt nào
thấy nó trên biểu đồ giá, nhưng nó là toàn bộ nội dung của giao dịch chênh lệch.

## 4 · Basis trong phiên — bước 1 phút

In [10]:
phien_cuoi = basis["date"].max()
ngay_str = phien_cuoi.strftime("%Y-%m-%d")

ib = client.intraday.derivative.basis(HD, start=ngay_str, end=ngay_str)
print(f"{len(ib)} thanh 1 phút · {ib['time'].min():%H:%M} → {ib['time'].max():%H:%M}")

fig = duong(
    ib,
    x="time",
    y="basis",
    tieu_de=f"{HD} — basis trong phiên {phien_cuoi:%d/%m/%Y}",
    phu_de="Bước 1 phút · basis co giãn liên tục trong phiên, không phải một con số cố định",
    nhan_y="điểm chỉ số",
    moc_khong=True,
)
fig.add_hline(y=ib["basis"].mean(), line_dash="dot", line_width=1, line_color="#898781",
              annotation_text=f"trung bình phiên {ib['basis'].mean():+.2f}", annotation_position="right")
fig

139 thanh 1 phút · 09:15 → 13:03


⚠️ Basis mở phiên và đóng phiên thường lệch nhau đáng kể — đó là lý do "basis
hôm nay bằng bao nhiêu" không có một câu trả lời duy nhất, và cũng là lý do
`basis()` EOD dùng **giá đóng cửa** và nói rõ điều đó qua tên cột
`future_close` / `spot_close`.

In [11]:
print(f"Basis đầu phiên : {ib['basis'].iloc[0]:+.2f} điểm")
print(f"Basis cuối phiên: {ib['basis'].iloc[-1]:+.2f} điểm")
print(f"Biên độ trong phiên: {ib['basis'].max() - ib['basis'].min():.2f} điểm")
print(f"Basis EOD ghi nhận: {basis[basis['date'] == phien_cuoi]['basis'].iloc[0]:+.2f} điểm")

Basis đầu phiên : +1.95 điểm
Basis cuối phiên: +4.08 điểm
Biên độ trong phiên: 9.62 điểm
Basis EOD ghi nhận: -1.82 điểm


## 5 · ⚠️ Phái sinh chỉ có HAI nhóm nhà đầu tư

Bốn nhóm chi tiết (cá nhân/tổ chức × trong nước/nước ngoài) **không tồn tại**
ở phái sinh. Kiểu của tham số chặn chuyện đó ngay lúc bạn gõ code — không phải
lúc chạy.

In [12]:
dong_ps = client.eod.derivative.investor.breakdown(HD, start=lui_ngay(HOM_NAY, thang=3))
print(f"Các nhóm ở phái sinh: {sorted(dong_ps['group'].unique())}")
print(f"Đơn vị khối lượng:   {dong_ps.attrs['finlens']['units']['buy_volume']}")

try:
    client.eod.derivative.investor.flow(HD, group="local_individual")
except (finlens.ValidationError, TypeError, ValueError) as e:
    print(f"\ngroup='local_individual' → {type(e).__name__}")
    print(f"  {str(e)[:200]}")

Các nhóm ở phái sinh: ['foreign', 'proprietary']
Đơn vị khối lượng:   contract



group='local_individual' → ValidationError
  [FL_VALIDATION] Các nhóm local_individual không có ở phái sinh. Nguồn của chúng (`eod_investor_trades`) khoá theo mã doanh nghiệp niêm yết, và không hợp đồng phái sinh nào có mặt trong danh mục đó — n


In [13]:
tich = (
    dong_ps.sort_values("date")
    .assign(tich_luy=lambda d: d.groupby("group", observed=True)["net_value"].cumsum() / 1e9)
)
ten_nhom = {"foreign": "Khối ngoại", "proprietary": "Tự doanh"}
tich["nhom"] = tich["group"].map(ten_nhom)

duong(
    tich,
    x="date",
    y="tich_luy",
    theo="nhom",
    tieu_de=f"{HD} — mua/bán ròng tích luỹ theo nhóm, 3 tháng",
    phu_de="Chỉ hai nhóm tồn tại ở phái sinh · giá trị tính bằng VND nên đổi sang tỷ đồng",
    nhan_y="tỷ đồng, tích luỹ",
    moc_khong=True,
)

## 6 · Basis có dự báo được chỉ số không?

Giả thuyết quen thuộc: basis âm sâu = thị trường bi quan = chỉ số sắp giảm.
Đo thẳng thay vì tin.

In [14]:
KHUNG = [1, 3, 5, 10]

do = basis.sort_values("date").copy()
for k in KHUNG:
    do[f"ls_{k}"] = (do["spot_close"].shift(-k) / do["spot_close"] - 1) * 100

# Chia basis_pct thành năm nhóm ngũ phân vị
do["nhom_basis"] = pd.qcut(do["basis_pct"], 5, labels=["Âm sâu nhất", "Q2", "Q3", "Q4", "Dương nhất"])

theo_nhom = do.groupby("nhom_basis", observed=True)[[f"ls_{k}" for k in KHUNG]].mean().round(3)
theo_nhom.columns = [f"{k} phiên" for k in KHUNG]
print("Lợi suất VN30 trung bình SAU ĐÓ, theo nhóm basis tại thời điểm quan sát (%):\n")
theo_nhom

Lợi suất VN30 trung bình SAU ĐÓ, theo nhóm basis tại thời điểm quan sát (%):



,1 phiên,3 phiên,5 phiên,10 phiên
nhom_basis,,,,
Âm sâu nhất,-0.172,0.130,0.048,0.514
Q2,0.196,0.071,-0.171,-0.384
Q3,0.024,0.345,0.922,1.087
Q4,0.113,0.073,-0.154,-0.172
Dương nhất,0.091,0.081,0.385,0.552


In [15]:
from finlens_examples import heatmap

heatmap(
    theo_nhom,
    tieu_de="Lợi suất VN30 sau đó, theo mức basis",
    phu_de=f"{len(do)} phiên · chia basis_pct thành 5 nhóm ngũ phân vị · đơn vị %",
    nhan_mau="%",
    dinh_dang_o="%{z:+.2f}",
)

### Đọc kết quả

Nếu basis có thông tin dự báo, các hàng phải xếp theo một chiều rõ ràng — nhóm
"âm sâu nhất" thấp hẳn, nhóm "dương nhất" cao hẳn. Bảng trên có xếp như vậy
không?

⚠️ Và một cảnh báo về cỡ mẫu: chia một năm dữ liệu thành năm nhóm là khoảng
**50 quan sát mỗi nhóm**. Với biến động ngày của VN30, chênh lệch vài phần mười
phần trăm giữa các nhóm nằm gọn trong sai số. Đây là một phép **thăm dò**,
không phải một phép kiểm định — muốn kết luận thì cần nhiều năm và một khoảng
tin cậy, đúng như cách notebook `33` làm.

In [16]:
n_moi_nhom = do.groupby("nhom_basis", observed=True).size()
sai_so_5p = do.groupby("nhom_basis", observed=True)["ls_5"].apply(lambda s: s.std() / np.sqrt(s.count()))
print("Cỡ mẫu và sai số chuẩn của lợi suất 5 phiên, theo nhóm:")
print(pd.DataFrame({"số quan sát": n_moi_nhom, "sai số chuẩn (%)": sai_so_5p.round(3)}).to_string())
print(f"\nBiên độ giữa nhóm cao nhất và thấp nhất ở khung 5 phiên: "
      f"{theo_nhom['5 phiên'].max() - theo_nhom['5 phiên'].min():.3f} điểm phần trăm")
print(f"Sai số chuẩn điển hình: {sai_so_5p.median():.3f} điểm phần trăm")

Cỡ mẫu và sai số chuẩn của lợi suất 5 phiên, theo nhóm:
             số quan sát  sai số chuẩn (%)
nhom_basis                                
Âm sâu nhất           50             0.410
Q2                    50             0.431
Q3                    50             0.403
Q4                    50             0.411
Dương nhất            50             0.467

Biên độ giữa nhóm cao nhất và thấp nhất ở khung 5 phiên: 1.093 điểm phần trăm
Sai số chuẩn điển hình: 0.411 điểm phần trăm


## 7 · Khối lượng chủ động của phái sinh

In [17]:
av = client.eod.derivative.active_volume(HD, start=lui_ngay(HOM_NAY, thang=2))
print(f"Đơn vị: {av.attrs['finlens']['units']['active_buy_volume']}  ← hợp đồng, không phải cổ phiếu")

av30 = av.tail(30).assign(nhan=lambda d: d["date"].dt.strftime("%d/%m"))
thanh_doi_mau(
    av30,
    x="nhan",
    y="net_active_volume",
    tieu_de=f"{HD} — khối lượng chủ động ròng, 30 phiên",
    phu_de="Đơn vị là HỢP ĐỒNG · nhớ rằng phần khớp định kỳ đã được chia đôi (notebook 41)",
    nhan_y="hợp đồng",
    dinh_dang_nhan="{:+,.0f}",
)

Đơn vị: contract  ← hợp đồng, không phải cổ phiếu


## Tổng kết

| Bạn cần | Gọi |
|---|---|
| Giá phái sinh | `eod.derivative.ohlcv("VN30F1M")` |
| Chênh lệch với cơ sở, theo phiên | `eod.derivative.basis("VN30F1M")` |
| Chênh lệch trong phiên | `intraday.derivative.basis("VN30F1M")` |
| Dòng tiền phái sinh | `eod.derivative.investor.flow(..., group="proprietary")` |

**Bốn điều mang sang notebook sau:**

1. **Đơn vị là `index_point` và `contract`.** Hệ số nhân (100.000 VND/điểm)
   **không nằm trong dữ liệu** — nó là đặc tả sản phẩm bạn phải tự biết.
2. **`basis_pct` lấy mẫu số là giá CHỈ SỐ.** Hai mẫu số lệch nhau chưa tới
   0,3% nên chọn nhầm không nhìn ra được — hỏi dữ liệu, đừng suy đoán.
3. **`basis()` không có `interval` là chủ ý.** Gộp một chênh lệch không có định
   nghĩa hiển nhiên; nếu bạn cần, hãy tự chọn phép gộp và nói rõ mình chọn gì.
4. Phái sinh **chỉ có hai nhóm** nhà đầu tư, và kiểu tham số chặn từ lúc gõ.

---

**Tiếp theo:** [`43_chung_quyen.ipynb`](43_chung_quyen.ipynb) — chứng quyền có
bảo đảm, và cái bẫy đơn vị 1000 lần.